# Forward Algorithm

The forward algorithm computes the probability of an observation sequence given an HMM. This is the **evaluation problem** in HMMs.

## Mathematical Background

The **forward variable** $\alpha_t(i)$ represents the probability of being in state $i$ at time $t$ and having observed the first $t$ observations:
$$\n\alpha_t(i) = P(q_1, ..., q_t = S_i, o_1, ..., o_t | \lambda)$$
### Recursion (Rabiner Eq. 32-33)

**Initialization** (t=1):$$\alpha_1(i) = \pi_i b_i(o_1)$$
**Induction** (t = 2, 3, ...):$$\alpha_t(j) = \left[ \sum_{i=1}^{N} \alpha_{t-1}(i) a_{ij} \right] b_j(o_t)$$
**Termination**:$$P(O|\lambda) = \sum_{i=1}^{N} \alpha_T(i)$$

In [ ]:
import numpy as np

from hmm import HMM, forward

## Basic Forward Algorithm

In [ ]:
# Create a simple HMM
A = np.array([[0.95, 0.05], [0.05, 0.95]])
B = np.array([[1/6]*6, [1/10]*5 + [1/2]])
V = [1, 2, 3, 4, 5, 6]

hmm = HMM(n_states=2, A=A, B=B, V=V)

# Run forward algorithm
obs = [1, 2, 1, 6, 6]
log_prob, alpha, c = forward(hmm, obs, scaling=True)

print(f"Observation sequence: {obs}")
print(f"Log probability: {log_prob:.4f}")
print(f"Probability: {np.exp(log_prob):.6f}")
print("\nAlpha matrix (forward variables):")
print(alpha)
print("\nScaling coefficients:")
print(c)

## Without Scaling

The algorithm works without scaling but can have numerical underflow for long sequences.

In [ ]:
# Without scaling - note: can have numerical issues for long sequences
prob, alpha = forward(hmm, obs, scaling=False)

print(f"Probability (without scaling): {prob:.6f}")
print("\nAlpha matrix:")
print(alpha)

## Scaling

Scaling normalizes alpha at each time step to prevent numerical underflow. The log probability is computed from the scaling coefficients.

In [ ]:
# Verify scaling works correctly
obs_long = [1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6]

log_prob_scaled, alpha_scaled, c_scaled = forward(hmm, obs_long, scaling=True)
prob_unscaled, alpha_unscaled = forward(hmm, obs_long, scaling=False)

print(f"Log probability (scaled): {log_prob_scaled:.4f}")
print(f"Log probability (unscaled): {np.log(prob_unscaled):.4f}")
print(f"\nScaling coefficients c[0:5]: {c_scaled[0:5]}")
print(f"Product of scaling coefficients: {np.prod(c_scaled):.2e}")

## Visualizing Scaling Coefficients

In [ ]:
import matplotlib.pyplot as plt

# Plot scaling coefficients over time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scaling coefficients
axes[0].stem(range(len(c_scaled)), c_scaled, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Scaling Coefficient c[t]')
axes[0].set_title('Scaling Coefficients (prevent underflow)')
axes[0].set_yscale('log')

# Forward probabilities over time
axes[1].plot(range(len(obs_long)), alpha_scaled[0, :], 'b-o', label='State 0', alpha=0.7)
axes[1].plot(range(len(obs_long)), alpha_scaled[1, :], 'r-o', label='State 1', alpha=0.7)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Alpha (scaled)')
axes[1].set_title('Forward Variables Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

- The forward algorithm computes P(O | λ) - probability of observation given model
- Use `scaling=True` for numerical stability with long sequences
- Returns log probability when scaling is enabled

Next notebook: [Viterbi Algorithm](./03-viterbi-algorithm.ipynb)